<a href="https://colab.research.google.com/github/natee-s/20-man-react-app-starter/blob/main/Math_VQA_Challenge2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: เชื่อมต่อ Google Drive เข้ากับ Colab
from google.colab import drive
drive.mount('/content/drive')

import os
# กำหนด Path ไปยังโฟลเดอร์ที่คุณสร้างทางลัดไว้ใน My Drive

data_dir = "/content/drive/MyDrive/4_Mylearning/7. Super AI Engineer/L.2/3. Hackathon/dataset"

# เช็คว่า Colab มองเห็นไฟล์ข้อมูลแล้วหรือยัง (ควรจะปริ้นท์ชื่อไฟล์อย่าง train.csv, test.csv ออกมา)
print("Files in dataset folder:", os.listdir(data_dir))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files in dataset folder: ['sample_submission.csv', 'train.csv', 'test.csv', 'images']


In [ ]:
# 3. ติดตั้งไลบรารีสำหรับการประมวลผล Vision-Language Model
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes qwen-vl-utils pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 24.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import os
from tqdm import tqdm
import re

# 1. กำหนด Path (ใช้ตัวแปรเดิมจาก Cell ด้านบน)
data_dir = "/content/drive/MyDrive/4_Mylearning/7. Super AI Engineer/L.2/3. Hackathon/dataset"

# 2. โหลดโมเดล Qwen2-VL-2B (แบบ 4-bit เพื่อไม่ให้ RAM ของ Colab T4 เต็ม)
print("Loading model... (อาจใช้เวลาสักครู่)")
model_id = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True
)
processor = AutoProcessor.from_pretrained(model_id)

# 3. โหลดไฟล์ CSV ของ Test set
test_csv_path = os.path.join(data_dir, "test.csv")
test_df = pd.read_csv(test_csv_path)

predictions = []

# 4. Prompt Engineering บังคับให้คิดเป็นขั้นเป็นตอนและตอบในแท็ก <answer>
system_prompt = """You are a highly capable Thai mathematics solver.
Read the problem from the image carefully. The problem may contain Thai text, equations, and diagrams.
Solve the problem step-by-step.
At the end, provide ONLY the final exact answer enclosed in <answer>...</answer> tags.
Do not include units if possible, just the number or expression.
For example: <answer>25</answer> or <answer>3sqrt3/2</answer>.
"""

print(f"Starting inference for {len(test_df)} images...")
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    # สร้าง Path สำหรับรูปภาพแต่ละข้อ
    # สมมติว่าใน test.csv คอลัมน์ชื่อ image_path เช่น "images/101_001.jpg"
    img_path = os.path.join(data_dir, row['image_path'])

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": system_prompt},
            ],
        }
    ]

    # เตรียม Input ให้โมเดล
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    # สั่งให้โมเดลประมวลผลและสร้างคำตอบ
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=512)

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    # 5. ดึงคำตอบออกมาจากแท็ก <answer>
    final_answer = ""
    if "<answer>" in output_text and "</answer>" in output_text:
        final_answer = output_text.split("<answer>")[1].split("</answer>")[0].strip()
    else:
        # กรณีโมเดลลืมใส่แท็ก ให้ดึงบรรทัดสุดท้ายมาเป็นคำตอบสำรอง
        final_answer = output_text.split("\n")[-1].strip()

    predictions.append({"id": row['id'], "answer": final_answer})

# 6. บันทึกผลลัพธ์ลงใน Google Drive
output_csv_path = os.path.join(data_dir, "submission.csv")
submission_df = pd.DataFrame(predictions)
submission_df.to_csv(output_csv_path, index=False)

print(f"\n✅ เสร็จสิ้น! ไฟล์คำตอบถูกบันทึกไว้ที่: {output_csv_path}")

Loading model... (อาจใช้เวลาสักครู่)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

TypeError: Qwen2VLForConditionalGeneration.__init__() got an unexpected keyword argument 'load_in_4bit'